# Dynamic Partition Pruning
`song_listened` is partitioned by `listen_date`. Joining it to a small date dimension that is filtered
should make Spark prune partitions at runtime (`dynamicpruning` in the plan). Cluster: 1 driver (this kernel) + 2 workers. Master UI 8080, worker UIs 8081/8082, app UI 4040 (Executors tab lists driver + executors).

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.master("spark://localhost:7077").appName("dynamic_partition_pruning")
         .config("spark.executor.memory", "1g").config("spark.cores.max", "2")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.uiWebUrl   # the Spark UI lives on this port (forwarded as 'Spark UI')

In [ ]:
song_df = (spark.read.option("header", "true").option("inferSchema", "true")
           .csv("data/songs.csv"))
song_listened_df = spark.read.parquet("data/song_listened")   # listen_date comes from the folder names
song_listened_df.printSchema()

In [ ]:
# small dimension keyed on the partition column
date_dim = (spark.sql("SELECT explode(sequence(to_date('2024-01-01'), to_date('2025-12-31'), interval 1 day)) AS listen_date")
            .withColumn("year_month", F.date_format("listen_date", "yyyy-MM")))

query = (song_listened_df.join(date_dim.filter("year_month = '2025-12'"), "listen_date")
         .groupBy("device").count())
query.explain()   # look for: PartitionFilters: [... dynamicpruningexpression(...)]

In [ ]:
query.show()   # then open the SQL tab in the Spark UI: 'number of partitions read' should be 31 of 731

In [ ]:
# same query with DPP off, for comparison in the UI
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "false")
query.show()
spark.conf.set("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true")